<a href="https://colab.research.google.com/github/YunaSon/TIL/blob/master/Python/concurrency.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 비동기 프로그래밍

## 1) 동시성(concurrency)

In [0]:
import time

def network_request(number):
    time.sleep(1.0)
    return {"success":True, "result":number**2}

In [0]:
def fetch_square(number):
    response = network_request(number)
    if response['success']:
      print("Result is:{}".format(response["result"]))

fetch_square(2)

Result is:4


## 2) 콜백(call back)

In [0]:
def wait_and_print(msg):
    time.sleep(1.0)
    print(msg)

In [0]:
import threading

def wait_and_print_async(msg):
    def callback():
        print(msg)
    timer = threading.Timer(1.0, callback)
    timer.start()

### 동기

In [0]:
wait_and_print("First call")
wait_and_print("Second call")
print("After call")

First call
Second call
After call


### 비동기

In [0]:
wait_and_print_async("First async")
wait_and_print_async("Second async")
print("After submission")

After submission


In [0]:
def network_request_async(number, on_done):
    def timer_done():
        on_done({"success":True,
                 "result":number**2})
    timer = threading.Timer(1.0, timer_done)
    timer.start()

In [0]:
def on_done(result):
    print(result)
  
network_request_async(2, on_done)

In [0]:
network_request_async(2, on_done)
network_request_async(3, on_done)
network_request_async(4, on_done)
print("After submission")

After submission


In [0]:
def fetch_square(number):
    def on_done(response):
        if response["success"]:
            print("Result is: {}".format(response['result']))
    network_request_async(number, on_done)

## 3) 퓨처

In [0]:
from concurrent.futures import Future

fut = Future()
fut

<Future at 0x7f730d046ac8 state=pending>

In [0]:
fut.set_result("Hello")

In [0]:
fut.result()

'Hello'

In [0]:
fut = Future()
fut.add_done_callback(lambda future: print(future.result(), flush=True))
fut.set_result("Hello")

Hello


In [0]:
def network_request_async(number):
    future = Future()
    result = {"success":True, "result": number**2}
    timer = threading.Timer(1.0, lambda: future.set_result(result))
    timer.start()
    return future

fut = network_request_async(2)

In [0]:
def fetch_square(number):
    fut = network_request_async(number)

    def on_done_future(future):
        response = future.result()
        if response["success"]:
            print("Result is:{}".format(response["result"]))

    fut.add_done_callback(on_done_future) 

## 4) 이벤트 루프

In [0]:
class Timer:

    def __init__(self, timeout):
        self.timeout = timeout
        self.start = time.time()

    def done(self):
        return time.time()

timer = Timer(1.0)

while True:
    if timer.done():
        print("Timer is done!")
        break

Timer is done!


In [0]:
class Timer:

    def __init__(self, timeout):
        self.timeout = timeout
        self.start = time.time()

    def done(self):
        return time.time()
    
    def on_timer_done(self, callback):
        self.callback = callback

timer = Timer(1.0)
timer.on_timer_done(lambda: print("Timer is done!"))

while True:
    if timer.done():
        timer.callback()
        break

Timer is done!


In [0]:
timers = []
timer1 = Timer(1.0)
timer1.on_timer_done(lambda: print("First timer is done!"))

timer2 = Timer(2.0)
timer2.on_timer_done(lambda: print("Second timer is done!"))

timers.append(timer1)
timers.append(timer2)

while True:
    for timer in timers:
        if timer.done():
            timer.callback()
            timers.remove(timer)
    if len(timers) == 0:
        break

First timer is done!
Second timer is done!


## 5) 코루틴

In [0]:
def range_generator(n):
    i = 0
    while i < n:
        print("Generating value {}".format(i))
        yield i
        i += 1

In [0]:
generator = range_generator(3)
generator

<generator object range_generator at 0x7f7318185db0>

In [0]:
next(generator)

Generating value 0


0

In [0]:
def parrot():
    while True:
        message = yield
        print("Parrot says: {}".format(message))

generator = parrot()
generator.send(None)
generator.send("Hello")
generator.send("World")

Parrot says: Hello
Parrot says: World
